In [1]:
# Run once if not installed
# pip install transformers torchaudio librosa accelerate

import torch
import torchaudio
import librosa
import numpy as np
import pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import WhisperModel, WhisperProcessor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report


c:\Users\Asus\anaconda3\envs\wav2vec_ser\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [3]:
df_train = pd.read_csv(r"D:\SER_Cross\data\processed\crema_d_metadata.csv")
df_test  = pd.read_csv(r"D:\SER_Cross\data\processed\ravdess_metadata.csv")

keep_emotions = ["angry", "happy", "sad"]

df_train = df_train[df_train["emotion"].isin(keep_emotions)]
df_test  = df_test[df_test["emotion"].isin(keep_emotions)]


In [4]:
le = LabelEncoder()
y_train = le.fit_transform(df_train["emotion"])
y_test  = le.transform(df_test["emotion"])

print("Classes:", le.classes_)


Classes: ['angry' 'happy' 'sad']


In [5]:
processor = WhisperProcessor.from_pretrained("openai/whisper-small")


c:\Users\Asus\anaconda3\envs\wav2vec_ser\lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
c:\Users\Asus\anaconda3\envs\wav2vec_ser\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Asus\.cache\huggingface\hub\models--openai--whisper-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python 

In [6]:
SAMPLE_RATE = 16000
MAX_LEN = 3 * SAMPLE_RATE  # 3 seconds

class WhisperSERDataset(Dataset):
    def __init__(self, paths, labels):
        self.paths = paths.reset_index(drop=True)
        self.labels = labels

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        wav, sr = torchaudio.load(self.paths[idx])
        wav = wav.mean(0)

        if sr != SAMPLE_RATE:
            wav = torchaudio.functional.resample(wav, sr, SAMPLE_RATE)

        if wav.shape[0] < MAX_LEN:
            wav = torch.nn.functional.pad(wav, (0, MAX_LEN - wav.shape[0]))
        else:
            wav = wav[:MAX_LEN]

        inputs = processor(
            wav.numpy(),
            sampling_rate=SAMPLE_RATE,
            return_tensors="pt"
        )

        return inputs.input_features.squeeze(0), self.labels[idx]


In [7]:
train_ds = WhisperSERDataset(df_train["path"], y_train)
test_ds  = WhisperSERDataset(df_test["path"], y_test)

train_loader = DataLoader(train_ds, batch_size=2, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=2, shuffle=False)


In [8]:
whisper = WhisperModel.from_pretrained("openai/whisper-small").to(device)

for p in whisper.parameters():
    p.requires_grad = False

print("Whisper loaded & frozen")


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Whisper loaded & frozen


In [9]:
class WhisperEmotionModel(nn.Module):
    def __init__(self, whisper):
        super().__init__()
        self.whisper = whisper
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(768, 3)

    def forward(self, x):
        outputs = self.whisper.encoder(x)
        hidden = outputs.last_hidden_state          # [B, T, 768]
        hidden = hidden.transpose(1, 2)
        pooled = self.pool(hidden).squeeze(-1)
        return self.fc(pooled)


In [10]:
model = WhisperEmotionModel(whisper).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)


In [11]:
from tqdm import tqdm

for epoch in range(2):
    model.train()
    correct, total = 0, 0

    for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        preds = logits.argmax(1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    print(f"Epoch {epoch+1} Train Accuracy:", correct / total)


Epoch 1:   0%|          | 0/1907 [00:01<?, ?it/s]


RuntimeError: "nll_loss_forward_reduce_cuda_kernel_2d_index" not implemented for 'Int'

In [ ]:
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        logits = model(x)
        preds = logits.argmax(1).cpu().numpy()

        y_pred.extend(preds)
        y_true.extend(y.numpy())

acc = accuracy_score(y_true, y_pred)
print("Whisper 3-Emotion Test Accuracy:", acc)

print(classification_report(
    y_true, y_pred,
    target_names=le.classes_
))
